# Synthetic Bubbles to Calibrate AI 

#### Wobbly Ellipse

In [5]:
import os
import glob
import csv
import cv2
import numpy as np

# ── Load ink-only character templates (0–9, +, F, I, m, s, dot, colon, space) ──
def load_char_templates(dirname="templates"):
    templates = {}
    for path in glob.glob(os.path.join(dirname, "*.png")):
        name = os.path.splitext(os.path.basename(path))[0]
        img  = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise FileNotFoundError(f"Missing template: {path}")
        # ensure RGBA
        if img.shape[2] == 3:
            bgr = img
            a   = np.ones(bgr.shape[:2], np.uint8) * 255
            img = np.dstack([bgr, a])
        templates[name] = img
    return templates

char_templates = load_char_templates("templates")

# map ts_txt chars → template keys
char_map = {
    ".":    "dot",
    ":":    "colon",
    " ":    "space"
}
# characters not in char_map (digits, '+','F','I','m','s') map to themselves

# ── CONFIG ────────────────────────────────────────────────────────────────
OUT_DIR    = "synthetic_bubble"
VID_NAME   = "ellipse_bubble_wobbly.mp4"
MASK_NAME  = "ellipse_bubble_wobbly_gt_mask.mp4"
GT_CSV     = "ellipse_bubble_wobbly_gt.csv"

FRAME_SIZE = (1280, 832)
NUM_FRAMES = 500
FPS        = 30

# ripple settings
NUM_PTS       = 8
EDGE_JITTER   = 64
JITTER_SMOOTH = 0.95
FINE_FACTOR   = 16

# base ellipse radii
a0, b0 = 25, 15

# bubble styling
BUBBLE_ALPHA = 0.8
BUBBLE_COLOR = (30, 30, 30)

# background
BKG_BRIGHT  = 230
NOISE_SIGMA = 3

# glare
GLARE_RADIUS  = 7
GLARE_BLUR    = 15
GLARE_MOVE_R  = 0.1
GLARE_ALPHA   = 0.9

# rise speed
VEL_PX_PER_FRAME = 2
VEL_PX_PER_SEC   = VEL_PX_PER_FRAME * FPS

# normalized bubble area
TARGET_AREA = np.pi * a0 * b0
CONST_DIAM  = 2 * np.sqrt(a0 * b0)

# ────────────────────────────────────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)

# MP4 writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

vid_out = cv2.VideoWriter(
    os.path.join(OUT_DIR, VID_NAME),
    fourcc,
    FPS,
    FRAME_SIZE
)

mask_out = cv2.VideoWriter(
    os.path.join(OUT_DIR, MASK_NAME),
    fourcc,
    FPS,
    FRAME_SIZE,
    isColor=False
)

w, h = FRAME_SIZE

# precompute for jitter
angles_base = np.linspace(0, 2*np.pi, NUM_PTS, endpoint=False)
jitter      = np.zeros(NUM_PTS, dtype=float)
num_fine    = NUM_PTS * FINE_FACTOR
angles_fine = np.linspace(0, 2*np.pi, num_fine, endpoint=False)

# ── kerning rules ─────────────────────────────
def kern(prev, nxt):
    if prev == ":" and nxt == "+":  return 5   # colon→plus gap
    if prev == "1":                 return 3   # digit '1' gets an extra px
    return 2

gt_rows = []

for i in range(NUM_FRAMES):
    # 1) background + noise
    frame = np.full((h, w, 3), BKG_BRIGHT, dtype=np.uint8)
    noise = np.random.normal(0, NOISE_SIGMA, (h, w, 3)).astype(np.int16)
    frame = np.clip(frame.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # 2) rising center
    cx  = w // 2
    cy0 = h + 100
    cy  = int(cy0 - VEL_PX_PER_FRAME * i)

    # 3) jitter update
    new_noise = np.random.normal(0, EDGE_JITTER*0.2, NUM_PTS)
    jitter    = JITTER_SMOOTH*jitter + (1-JITTER_SMOOTH)*new_noise
    jitter    = np.clip(jitter, -EDGE_JITTER, EDGE_JITTER)

    # 4) smooth by interpolation
    ang_ext     = np.concatenate([angles_base, angles_base+2*np.pi])
    jit_ext     = np.concatenate([jitter, jitter])
    jitter_fine = np.interp(angles_fine, ang_ext, jit_ext)

    # 5) build raw ellipse contour
    radii_major = a0 + jitter_fine
    radii_minor = b0 + jitter_fine
    xs = cx + radii_major * np.cos(angles_fine)
    ys = cy + radii_minor * np.sin(angles_fine)
    pts = np.stack([xs, ys], axis=-1).astype(np.int32)[None, ...]

    # 6) normalize area
    contour      = pts[0].reshape(-1,1,2).astype(np.float32)
    current_area = cv2.contourArea(contour)
    if current_area > 1e-3:
        scale = np.sqrt(TARGET_AREA / current_area)
        xs = cx + (xs - cx)*scale
        ys = cy + (ys - cy)*scale
        pts = np.stack([xs, ys], axis=-1).astype(np.int32)[None, ...]

    # 7) composite bubble (and create GT mask)
    mask = np.zeros((h, w), np.uint8)
    cv2.fillPoly(mask, pts, 255)

    # write GT mask frame to mask video
    mask_out.write(mask)

    # alpha composite bubble into frame
    alpha3 = (mask.astype(float)/255.0)[...,None] * BUBBLE_ALPHA
    layer  = np.full_like(frame, BUBBLE_COLOR)
    frame  = (frame*(1-alpha3) + layer*alpha3).astype(np.uint8)

    # 8) add glare
    ang_g  = 2*np.pi * (i/NUM_FRAMES)
    gx     = int(cx + a0*GLARE_MOVE_R*np.cos(ang_g))
    gy     = int(cy + b0*GLARE_MOVE_R*np.sin(ang_g))
    glare  = np.zeros((h, w), np.uint8)
    cv2.circle(glare, (gx,gy), GLARE_RADIUS, 255, -1)
    glare  = cv2.GaussianBlur(glare, (GLARE_BLUR,GLARE_BLUR), 0)
    ga3    = (glare.astype(float)/255.0)[...,None] * GLARE_ALPHA
    frame  = (frame*(1-ga3) + 255*ga3).astype(np.uint8)

    # 9) annotate bubble stats
    info_txt = f"ID 0   D={CONST_DIAM:.1f}px   V={VEL_PX_PER_SEC:.1f}px/s"
    cv2.putText(frame, info_txt, (10,30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                (0,255,0), 2, cv2.LINE_AA)

    # 10) black bar
    bar_h = 30
    cv2.rectangle(frame, (0,h-bar_h), (w,h), (0,0,0), -1)

    # 11) exact kerning-aware Phantom timestamp
    time_ms = i * (1000.0 / FPS)
    ts_txt  = f"FI+: +{time_ms:0.3f} ms"
    chars   = [c for c in ts_txt if c != " "]  # drop spaces

    widths   = [char_templates[char_map.get(c, c)].shape[1] for c in chars]
    spacings = [kern(chars[j], chars[j+1]) for j in range(len(chars)-1)]
    Wts      = int(sum(widths) + sum(spacings))
    Hts      = int(max(char_templates[char_map.get(c, c)].shape[0] for c in chars))

    overlay = np.zeros((Hts, Wts, 4), dtype=np.uint8)
    x_off   = 0
    for j, c in enumerate(chars):
        key = char_map.get(c, c)
        tpl = char_templates[key]
        h_, w_ = tpl.shape[:2]
        overlay[0:h_, x_off:x_off+w_] = tpl
        x_off += w_
        if j < len(chars)-1:
            x_off += kern(c, chars[j+1])

    X_ts  = 5
    Y_ts  = h - bar_h + (bar_h - Hts)//2

    alpha = overlay[...,3:4].astype(float)/255.0
    roi   = frame[Y_ts:Y_ts+Hts, X_ts:X_ts+Wts].astype(float)
    comp  = roi*(1-alpha) + overlay[...,:3].astype(float)*alpha
    frame[Y_ts:Y_ts+Hts, X_ts:X_ts+Wts] = comp.astype(np.uint8)

    # ── GT CSV metrics ───────────────────────
    M = cv2.moments(mask, binaryImage=True)
    if M["m00"] > 0:
        gt_cx = M["m10"] / M["m00"]
        gt_cy = M["m01"] / M["m00"]
        area_px = int(cv2.countNonZero(mask))
        diam_px = float(np.sqrt(4.0 * area_px / np.pi))

        ys_nz, xs_nz = np.where(mask > 0)
        xmin, xmax = int(xs_nz.min()), int(xs_nz.max())
        ymin, ymax = int(ys_nz.min()), int(ys_nz.max())
    else:
        gt_cx = gt_cy = float("nan")
        area_px = 0
        diam_px = float("nan")
        xmin = ymin = xmax = ymax = -1

    gt_rows.append([
        i, i / FPS, 0,
        gt_cx, gt_cy,
        area_px, diam_px,
        0.0, -float(VEL_PX_PER_SEC), float(VEL_PX_PER_SEC),
        xmin, ymin, xmax, ymax
    ])

    # write frame
    vid_out.write(frame)

    if (i+1) % 50 == 0:
        print(f"frame {i+1}/{NUM_FRAMES}")

vid_out.release()
mask_out.release()

# write GT CSV
gt_csv_path = os.path.join(OUT_DIR, GT_CSV)
with open(gt_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "frame","time_s","gt_id",
        "gt_cx_px","gt_cy_px",
        "gt_area_px2","gt_diam_equiv_px",
        "gt_vx_px_s","gt_vy_px_s","gt_vel_px_s",
        "gt_bbox_xmin","gt_bbox_ymin","gt_bbox_xmax","gt_bbox_ymax"
    ])
    writer.writerows(gt_rows)

print("Done.")
print("Video:", os.path.join(OUT_DIR, VID_NAME))
print("GT mask video:", os.path.join(OUT_DIR, MASK_NAME))
print("GT CSV:", gt_csv_path)


frame 50/500
frame 100/500
frame 150/500
frame 200/500
frame 250/500
frame 300/500
frame 350/500
frame 400/500
frame 450/500
frame 500/500
Done.
Video: synthetic_bubble\ellipse_bubble_wobbly.mp4
GT mask video: synthetic_bubble\ellipse_bubble_wobbly_gt_mask.mp4
GT CSV: synthetic_bubble\ellipse_bubble_wobbly_gt.csv


#### Perfect Ellipse

In [6]:
import os
import glob
import csv
import cv2
import numpy as np

# ── Load your ink-only character templates ────────────────────────────────
def load_char_templates(dirname="templates"):
    templates = {}
    for path in glob.glob(os.path.join(dirname, "*.png")):
        name = os.path.splitext(os.path.basename(path))[0]
        img  = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise FileNotFoundError(f"Missing template: {path}")
        if img.shape[2] == 3:  # add alpha if needed
            bgr = img
            a   = np.ones(bgr.shape[:2], np.uint8) * 255
            img = np.dstack([bgr, a])
        templates[name] = img
    return templates

char_templates = load_char_templates("templates")

# map timestamp chars → template names
char_map = {".": "dot", ":": "colon", " ": "space"}

# ── CONFIG ───────────────────────────────────────────────────────────────
OUT_DIR     = "synthetic_bubble"
VID_NAME    = "ellipse_bubble.mp4"
MASK_NAME   = "ellipse_bubble_gt_mask.mp4"
GT_CSV_NAME = "ellipse_bubble_gt.csv"

FRAME_SIZE = (1280, 832)
NUM_FRAMES = 500
FPS        = 30

# bubble geometry
a0, b0 = 25, 15              # semi-major, semi-minor
CONST_DIAM  = 2 * np.sqrt(a0 * b0)      # equivalent diameter (by geometric mean)
TARGET_AREA = np.pi * a0 * b0

# styling
BUBBLE_ALPHA = 0.8
BUBBLE_COLOR = (30, 30, 30)
BKG_BRIGHT   = 230
NOISE_SIGMA  = 3

# glare
GLARE_RADIUS  = 7
GLARE_BLUR    = 15
GLARE_MOVE_R  = 0.1
GLARE_ALPHA   = 0.9

# motion
VEL_PF  = 2
VEL_PS  = VEL_PF * FPS

# ── setup writers (MP4) ──────────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

vid_out = cv2.VideoWriter(
    os.path.join(OUT_DIR, VID_NAME), fourcc, FPS, FRAME_SIZE
)

mask_out = cv2.VideoWriter(
    os.path.join(OUT_DIR, MASK_NAME), fourcc, FPS, FRAME_SIZE, isColor=False
)

w, h = FRAME_SIZE

# precompute angles for a smooth ellipse
angles = np.linspace(0, 2*np.pi, 360, endpoint=False)

# ── kerning rules  ────────────────────────────────
def kern(prev, nxt):
    if prev == ":" and nxt == "+":  return 5   # colon→plus gap
    if prev == "1":                 return 3   # digit '1' extra px
    return 2

gt_rows = []

for i in range(NUM_FRAMES):
    # 1) background + noise
    frame = np.full((h, w, 3), BKG_BRIGHT, dtype=np.uint8)
    noise = np.random.normal(0, NOISE_SIGMA, (h, w, 3)).astype(np.int16)
    frame = np.clip(frame.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # 2) rising center
    cx  = w // 2
    cy0 = h + 100
    cy  = int(cy0 - VEL_PF * i)

    # 5) build perfect ellipse contour
    xs = cx + a0 * np.cos(angles)
    ys = cy + b0 * np.sin(angles)
    pts = np.stack([xs, ys], axis=-1).astype(np.int32)[None, ...]

    # 6) normalize area 
    contour      = pts[0].reshape(-1, 1, 2).astype(np.float32)
    current_area = cv2.contourArea(contour)
    if current_area > 1e-3:
        scale = np.sqrt(TARGET_AREA / current_area)
        xs = cx + (xs - cx) * scale
        ys = cy + (ys - cy) * scale
        pts = np.stack([xs, ys], axis=-1).astype(np.int32)[None, ...]

    # 7) composite bubble + GT mask
    mask = np.zeros((h, w), np.uint8)
    cv2.fillPoly(mask, pts, 255)

    # save GT mask video frame
    mask_out.write(mask)

    alpha3 = (mask.astype(float) / 255.0)[..., None] * BUBBLE_ALPHA
    layer  = np.full_like(frame, BUBBLE_COLOR)
    frame  = (frame * (1 - alpha3) + layer * alpha3).astype(np.uint8)

    # 8) glare
    ang_g = 2*np.pi * (i / NUM_FRAMES)
    gx    = int(cx + a0 * GLARE_MOVE_R * np.cos(ang_g))
    gy    = int(cy + b0 * GLARE_MOVE_R * np.sin(ang_g))
    glare = np.zeros((h, w), np.uint8)
    cv2.circle(glare, (gx, gy), GLARE_RADIUS, 255, -1)
    glare = cv2.GaussianBlur(glare, (GLARE_BLUR, GLARE_BLUR), 0)
    ga3   = (glare.astype(float) / 255.0)[..., None] * GLARE_ALPHA
    frame = (frame * (1 - ga3) + 255 * ga3).astype(np.uint8)

    # 9) annotate bubble stats
    info_txt = f"ID 0   D={CONST_DIAM:.1f}px   V={VEL_PS:.1f}px/s"
    cv2.putText(frame, info_txt, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                (0, 255, 0), 2, cv2.LINE_AA)

    # 10) black bar
    bar_h = 30
    cv2.rectangle(frame, (0, h - bar_h), (w, h), (0, 0, 0), -1)

    # 11) exact kerning-aware Phantom timestamp (template-based)
    time_ms = i * (1000.0 / FPS)
    ts_txt  = f"FI+: +{time_ms:0.3f} ms"
    chars   = [c for c in ts_txt if c != " "]  # drop spaces

    widths   = [char_templates[char_map.get(c, c)].shape[1] for c in chars]
    spacings = [kern(chars[j], chars[j+1]) for j in range(len(chars) - 1)]
    Wts      = int(sum(widths) + sum(spacings))
    Hts      = int(max(char_templates[char_map.get(c, c)].shape[0] for c in chars))

    overlay = np.zeros((Hts, Wts, 4), dtype=np.uint8)
    x_off   = 0
    for j, c in enumerate(chars):
        key = char_map.get(c, c)
        tpl = char_templates[key]
        h_, w_ = tpl.shape[:2]
        overlay[0:h_, x_off:x_off + w_] = tpl
        x_off += w_
        if j < len(chars) - 1:
            x_off += kern(c, chars[j+1])

    X_ts = 5
    Y_ts = h - bar_h + (bar_h - Hts) // 2

    alpha = overlay[..., 3:4].astype(float) / 255.0
    roi   = frame[Y_ts:Y_ts + Hts, X_ts:X_ts + Wts].astype(float)
    comp  = roi * (1 - alpha) + overlay[..., :3].astype(float) * alpha
    frame[Y_ts:Y_ts + Hts, X_ts:X_ts + Wts] = comp.astype(np.uint8)

    # ── GT CSV metrics (safe when bubble off-screen) ───────────────────────
    M = cv2.moments(mask, binaryImage=True)
    if M["m00"] > 0:
        gt_cx = M["m10"] / M["m00"]
        gt_cy = M["m01"] / M["m00"]
        area_px = int(cv2.countNonZero(mask))
        diam_px = float(np.sqrt(4.0 * area_px / np.pi))

        ys_nz, xs_nz = np.where(mask > 0)
        xmin, xmax = int(xs_nz.min()), int(xs_nz.max())
        ymin, ymax = int(ys_nz.min()), int(ys_nz.max())
    else:
        gt_cx = gt_cy = float("nan")
        area_px = 0
        diam_px = float("nan")
        xmin = ymin = xmax = ymax = -1

    gt_rows.append([
        i, i / FPS, 0,
        gt_cx, gt_cy,
        area_px, diam_px,
        0.0, -float(VEL_PS), float(VEL_PS),
        xmin, ymin, xmax, ymax
    ])

    # write + progress
    vid_out.write(frame)
    if (i + 1) % 50 == 0:
        print(f"frame {i+1}/{NUM_FRAMES}")

vid_out.release()
mask_out.release()

# write GT CSV
gt_csv_path = os.path.join(OUT_DIR, GT_CSV_NAME)
with open(gt_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "frame","time_s","gt_id",
        "gt_cx_px","gt_cy_px",
        "gt_area_px2","gt_diam_equiv_px",
        "gt_vx_px_s","gt_vy_px_s","gt_speed_px_s",
        "gt_bbox_xmin","gt_bbox_ymin","gt_bbox_xmax","gt_bbox_ymax"
    ])
    writer.writerows(gt_rows)

print("Done.")
print("Video:", os.path.join(OUT_DIR, VID_NAME))
print("GT mask video:", os.path.join(OUT_DIR, MASK_NAME))
print("GT CSV:", gt_csv_path)


frame 50/500
frame 100/500
frame 150/500
frame 200/500
frame 250/500
frame 300/500
frame 350/500
frame 400/500
frame 450/500
frame 500/500
Done.
Video: synthetic_bubble\ellipse_bubble.mp4
GT mask video: synthetic_bubble\ellipse_bubble_gt_mask.mp4
GT CSV: synthetic_bubble\ellipse_bubble_gt.csv


#### Perfect Circle

In [8]:
import os
import glob
import csv
import cv2
import numpy as np

# ── Load your ink-only character templates ────────────────────────────────
def load_char_templates(dirname="templates"):
    templates = {}
    for path in glob.glob(os.path.join(dirname, "*.png")):
        name = os.path.splitext(os.path.basename(path))[0]
        img  = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise FileNotFoundError(f"Missing template: {path}")
        if img.shape[2] == 3:  # add alpha if needed
            bgr = img
            a   = np.ones(bgr.shape[:2], np.uint8) * 255
            img = np.dstack([bgr, a])
        templates[name] = img
    return templates

char_templates = load_char_templates("templates")
char_map = {".": "dot", ":": "colon"}  # spaces dropped later

# ── CONFIG ───────────────────────────────────────────────────────────────
OUT_DIR     = "synthetic_bubble"
VID_NAME    = "circle_bubble.mp4"
MASK_NAME   = "circle_bubble_gt_mask.mp4"
GT_CSV_NAME = "circle_bubble_gt.csv"

FRAME_SIZE = (1280, 832)
NUM_FRAMES = 500
FPS        = 30

# circle geometry — diameter = 40 px
CONST_DIAM = 40.0
RADIUS     = CONST_DIAM / 2.0

# styling
BUBBLE_ALPHA = 0.8
BUBBLE_COLOR = (30, 30, 30)
BKG_BRIGHT   = 230
NOISE_SIGMA  = 3

# glare
GLARE_RADIUS  = 7
GLARE_BLUR    = 15
GLARE_MOVE_R  = 0.1
GLARE_ALPHA   = 0.9

# motion
VEL_PF = 2                 # px/frame upward
VEL_PS = VEL_PF * FPS      # px/s

# ── setup video writers (MP4) ────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

vid_out = cv2.VideoWriter(
    os.path.join(OUT_DIR, VID_NAME), fourcc, FPS, FRAME_SIZE
)
mask_out = cv2.VideoWriter(
    os.path.join(OUT_DIR, MASK_NAME), fourcc, FPS, FRAME_SIZE, isColor=False
)

w, h = FRAME_SIZE

# ── kerning rules ────────────────────────────────
def kern(prev, nxt):
    if prev == ":" and nxt == "+":  return 5   
    if prev == "1":                 return 3   
    return 2

gt_rows = []

for i in range(NUM_FRAMES):
    # 1) background + noise
    frame = np.full((h, w, 3), BKG_BRIGHT, dtype=np.uint8)
    noise = np.random.normal(0, NOISE_SIGMA, (h, w, 3)).astype(np.int16)
    frame = np.clip(frame.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # 2) rising center
    cx  = w // 2
    cy0 = h + 100
    cy  = int(cy0 - VEL_PF * i)

    # 3) perfect circle mask
    mask = np.zeros((h, w), np.uint8)
    cv2.circle(mask, (cx, cy), int(round(RADIUS)), 255, -1)

    # save GT mask video frame
    mask_out.write(mask)

    # 4) composite bubble
    alpha3 = (mask.astype(float) / 255.0)[..., None] * BUBBLE_ALPHA
    layer  = np.full_like(frame, BUBBLE_COLOR)
    frame  = (frame * (1 - alpha3) + layer * alpha3).astype(np.uint8)

    # 5) glare moving around circle
    ang_g = 2 * np.pi * (i / NUM_FRAMES)
    gx    = int(cx + RADIUS * GLARE_MOVE_R * np.cos(ang_g))
    gy    = int(cy + RADIUS * GLARE_MOVE_R * np.sin(ang_g))
    glare = np.zeros((h, w), np.uint8)
    cv2.circle(glare, (gx, gy), GLARE_RADIUS, 255, -1)
    glare = cv2.GaussianBlur(glare, (GLARE_BLUR, GLARE_BLUR), 0)
    ga3   = (glare.astype(float) / 255.0)[..., None] * GLARE_ALPHA
    frame = (frame * (1 - ga3) + 255 * ga3).astype(np.uint8)

    # 6) annotate bubble stats
    info_txt = f"ID 0   D={CONST_DIAM:.1f}px   V={VEL_PS:.1f}px/s"
    cv2.putText(frame, info_txt, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                (0, 255, 0), 2, cv2.LINE_AA)

    # 7) black bar for timestamp
    bar_h = 30
    cv2.rectangle(frame, (0, h - bar_h), (w, h), (0, 0, 0), -1)

    # 8) exact kerning-aware Phantom timestamp
    time_ms = i * (1000.0 / FPS)
    ts_txt  = f"FI+: +{time_ms:0.3f} ms"
    chars   = [c for c in ts_txt if c != " "]  # drop spaces

    widths   = [char_templates[char_map.get(c, c)].shape[1] for c in chars]
    spacings = [kern(chars[j], chars[j+1]) for j in range(len(chars) - 1)]
    Wts      = int(sum(widths) + sum(spacings))
    Hts      = int(max(char_templates[char_map.get(c, c)].shape[0] for c in chars))

    overlay = np.zeros((Hts, Wts, 4), dtype=np.uint8)
    x_off   = 0
    for j, c in enumerate(chars):
        key = char_map.get(c, c)
        tpl = char_templates[key]
        h_, w_ = tpl.shape[:2]
        overlay[0:h_, x_off:x_off + w_] = tpl
        x_off += w_
        if j < len(chars) - 1:
            x_off += kern(c, chars[j+1])

    X_ts = 5
    Y_ts = h - bar_h + (bar_h - Hts) // 2

    alpha = overlay[..., 3:4].astype(float) / 255.0
    roi   = frame[Y_ts:Y_ts + Hts, X_ts:X_ts + Wts].astype(float)
    comp  = roi * (1 - alpha) + overlay[..., :3].astype(float) * alpha
    frame[Y_ts:Y_ts + Hts, X_ts:X_ts + Wts] = comp.astype(np.uint8)

    # ── GT CSV metrics (safe when bubble off-screen) ───────────────────────
    M = cv2.moments(mask, binaryImage=True)
    if M["m00"] > 0:
        gt_cx = M["m10"] / M["m00"]
        gt_cy = M["m01"] / M["m00"]
        area_px = int(cv2.countNonZero(mask))
        diam_px = float(np.sqrt(4.0 * area_px / np.pi))

        ys_nz, xs_nz = np.where(mask > 0)
        xmin, xmax = int(xs_nz.min()), int(xs_nz.max())
        ymin, ymax = int(ys_nz.min()), int(ys_nz.max())
    else:
        gt_cx = gt_cy = float("nan")
        area_px = 0
        diam_px = float("nan")
        xmin = ymin = xmax = ymax = -1

    gt_rows.append([
        i, i / FPS, 0,
        gt_cx, gt_cy,
        area_px, diam_px,
        0.0, -float(VEL_PS), float(VEL_PS),
        xmin, ymin, xmax, ymax
    ])

    # 9) write frame + progress
    vid_out.write(frame)
    if (i + 1) % 50 == 0:
        print(f"frame {i+1}/{NUM_FRAMES}")

vid_out.release()
mask_out.release()

# write GT CSV
gt_csv_path = os.path.join(OUT_DIR, GT_CSV_NAME)
with open(gt_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "frame","time_s","gt_id",
        "gt_cx_px","gt_cy_px",
        "gt_area_px2","gt_diam_equiv_px",
        "gt_vx_px_s","gt_vy_px_s","gt_speed_px_s",
        "gt_bbox_xmin","gt_bbox_ymin","gt_bbox_xmax","gt_bbox_ymax"
    ])
    writer.writerows(gt_rows)

print("Done.")
print("Video:", os.path.join(OUT_DIR, VID_NAME))
print("GT mask video:", os.path.join(OUT_DIR, MASK_NAME))
print("GT CSV:", gt_csv_path)


frame 50/500
frame 100/500
frame 150/500
frame 200/500
frame 250/500
frame 300/500
frame 350/500
frame 400/500
frame 450/500
frame 500/500
Done.
Video: synthetic_bubble\circle_bubble.mp4
GT mask video: synthetic_bubble\circle_bubble_gt_mask.mp4
GT CSV: synthetic_bubble\circle_bubble_gt.csv


#### Spinning Bubble

In [5]:
import os
import glob
import csv
import cv2
import numpy as np

# ── Load your ink-only character templates ────────────────────────────────
def load_char_templates(dirname="templates"):
    templates = {}
    for path in glob.glob(os.path.join(dirname, "*.png")):
        name = os.path.splitext(os.path.basename(path))[0]
        img  = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise FileNotFoundError(f"Missing template: {path}")
        # ensure RGBA
        if img.shape[2] == 3:
            bgr = img
            a   = np.ones(bgr.shape[:2], np.uint8) * 255
            img = np.dstack([bgr, a])
        templates[name] = img
    return templates

char_templates = load_char_templates("templates")

# map timestamp chars → template keys
char_map = {
    ".": "dot",
    ":": "colon",
    " ": "space",
}

# ── CONFIG ────────────────────────────────────────────────────────────────
# apparatus geometry
APP_W = 700
APP_H = 360
APP_BORDER = 18
CENTER_BAR_W = 80
WINDOW_GAP = 24

# burst offset behavior
BURST_OFFSET_MAX = 18   # px, apparatus shifts left/right once per burst

# horizontal bubble motion
HORIZ_SPEED_PX_S = 55.0
OUT_DIR     = "synthetic_bubble"
BASE_NAME   = "ellipse_bubble_wobbly_spin_bursts_two_bubbles"

VID_NAME    = f"{BASE_NAME}.mp4"
MASK_NAME   = f"{BASE_NAME}_gt_mask.mp4"
GT_CSV_NAME = f"{BASE_NAME}_gt.csv"

FRAME_SIZE = (1280, 832)
NUM_FRAMES = 500
FPS        = 30

# burst behavior (spin-mode emulation)
BURST_FRAMES  = 8
DT_IN_MS      = 1000.0 / FPS
GAP_FACTOR    = 3.0
DT_GAP_MS     = DT_IN_MS * GAP_FACTOR

# second bubble timing (seconds after first starts)
SECOND_START_T = 4.0

# ripple settings
NUM_PTS       = 8
EDGE_JITTER   = 64
JITTER_SMOOTH = 0.95
FINE_FACTOR   = 16

# base ellipse radii
a0, b0 = 25, 15

# bubble styling
BUBBLE_ALPHA = 0.8
BUBBLE_COLOR = (30, 30, 30)

# background
BKG_BRIGHT  = 230
NOISE_SIGMA = 3

# glare
GLARE_RADIUS = 7
GLARE_BLUR   = 15
GLARE_MOVE_R = 0.1
GLARE_ALPHA  = 0.9

# rise speed (px/s)
VEL_PX_PER_SEC = 60.0

# normalized bubble area
TARGET_AREA = np.pi * a0 * b0
CONST_DIAM  = 2 * np.sqrt(a0 * b0)

# ── setup writers (MP4) ──────────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

vid_out = cv2.VideoWriter(
    os.path.join(OUT_DIR, VID_NAME), fourcc, FPS, FRAME_SIZE
)
mask_out = cv2.VideoWriter(
    os.path.join(OUT_DIR, MASK_NAME), fourcc, FPS, FRAME_SIZE, isColor=False
)

w, h = FRAME_SIZE

NUM_BURSTS = int(np.ceil(NUM_FRAMES / BURST_FRAMES))
burst_offsets = np.array([
    -BURST_OFFSET_MAX if (k % 2 == 0) else BURST_OFFSET_MAX
    for k in range(NUM_BURSTS)
], dtype=int)

# precompute for jitter
angles_base = np.linspace(0, 2*np.pi, NUM_PTS, endpoint=False)
jitter      = np.zeros(NUM_PTS, dtype=float)

num_fine    = NUM_PTS * FINE_FACTOR
angles_fine = np.linspace(0, 2*np.pi, num_fine, endpoint=False)

# ── kerning rules ────────────────────────────────
def kern(prev, nxt):
    if prev == ":" and nxt == "+":  return 5
    if prev == "1":                 return 3
    return 2

# ── helper: compute GT stats from a binary mask ───────────────────────────
def gt_stats_from_mask(mask_u8):
    M = cv2.moments(mask_u8, binaryImage=True)
    if M["m00"] <= 0:
        return {
            "cx": float("nan"),
            "cy": float("nan"),
            "area": 0,
            "diam": float("nan"),
            "xmin": -1, "ymin": -1, "xmax": -1, "ymax": -1
        }
    cx = M["m10"] / M["m00"]
    cy = M["m01"] / M["m00"]
    area_px = int(cv2.countNonZero(mask_u8))
    diam_px = float(np.sqrt(4.0 * area_px / np.pi))

    ys, xs = np.where(mask_u8 > 0)
    xmin, xmax = int(xs.min()), int(xs.max())
    ymin, ymax = int(ys.min()), int(ys.max())
    return {
        "cx": cx, "cy": cy,
        "area": area_px,
        "diam": diam_px,
        "xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax
    }

# running timestamp in ms (Phantom overlay)
time_ms = 0.0

gt_rows = []

for i in range(NUM_FRAMES):
    # burst index and in-burst index
    bidx, ib = divmod(i, BURST_FRAMES)

    # update timestamp with burst behavior
    if i == 0:
        time_ms = 0.0
    else:
        time_ms += DT_GAP_MS if ib == 0 else DT_IN_MS

    # 1) background + noise
    frame = np.full((h, w, 3), BKG_BRIGHT, dtype=np.uint8)
    noise = np.random.normal(0, NOISE_SIGMA, (h, w, 3)).astype(np.int16)
    frame = np.clip(frame.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    burst_dx = int(burst_offsets[bidx])

    app_cx = w // 2 + burst_dx
    app_cy = h // 2

    app_x0 = app_cx - APP_W // 2
    app_y0 = app_cy - APP_H // 2
    app_x1 = app_x0 + APP_W
    app_y1 = app_y0 + APP_H

    # center bar
    bar_x0 = app_cx - CENTER_BAR_W // 2
    bar_x1 = bar_x0 + CENTER_BAR_W

    # two square-ish windows
    usable_w = APP_W - 2 * APP_BORDER - CENTER_BAR_W - WINDOW_GAP
    win_w = usable_w // 2
    win_h = APP_H - 2 * APP_BORDER

    left_win_x0 = app_x0 + APP_BORDER
    left_win_x1 = left_win_x0 + win_w

    right_win_x1 = app_x1 - APP_BORDER
    right_win_x0 = right_win_x1 - win_w

    win_y0 = app_y0 + APP_BORDER
    win_y1 = win_y0 + win_h
    
    # apparatus body
    cv2.rectangle(frame, (app_x0, app_y0), (app_x1, app_y1), (80, 80, 80), -1)

    # two bright windows
    cv2.rectangle(frame, (left_win_x0, win_y0), (left_win_x1, win_y1), (235, 235, 235), -1)
    cv2.rectangle(frame, (right_win_x0, win_y0), (right_win_x1, win_y1), (235, 235, 235), -1)

    # borders around apparatus and windows
    cv2.rectangle(frame, (app_x0, app_y0), (app_x1, app_y1), (30, 30, 30), 3)
    cv2.rectangle(frame, (left_win_x0, win_y0), (left_win_x1, win_y1), (40, 40, 40), 2)
    cv2.rectangle(frame, (right_win_x0, win_y0), (right_win_x1, win_y1), (40, 40, 40), 2)

    # center bar visible through the middle
    cv2.rectangle(frame, (bar_x0, 0), (bar_x1, h - 50), (55, 55, 55), -1)
    cv2.rectangle(frame, (bar_x0, 0), (bar_x1, h - 50), (20, 20, 20), 2)

    # 2) apparatus position (constant for entire burst)

    # bubble centers move horizontally in apparatus coordinates
    t1_s = time_ms / 1000.0
    t2_s = t1_s - SECOND_START_T
    has_bubble2 = t2_s >= 0.0

    cy1 = (win_y0 + win_y1) // 2
    cy2 = (win_y0 + win_y1) // 2 if has_bubble2 else None

    # keep bubbles inside windows with margin
    x_margin = 60
    left_center_x = (left_win_x0 + left_win_x1) // 2
    right_center_x = (right_win_x0 + right_win_x1) // 2
    x_amp = max(20, (win_w // 2) - x_margin)

    # continuous motion from TRUE time
    rel1 = x_amp * np.sin(2 * np.pi * 0.18 * t1_s)
    rel2 = x_amp * np.sin(2 * np.pi * 0.18 * t2_s) if has_bubble2 else None

    # camera sees opposite side every 180 degrees (every burst)
    # use MIRRORED offset in opposite window
    if bidx % 2 == 0:
        cx1 = int(left_center_x + rel1)
        cx2 = int(right_center_x + rel2) if has_bubble2 else None
    else:
        cx1 = int(right_center_x - rel1)
        cx2 = int(left_center_x - rel2) if has_bubble2 else None

    # 3) jitter update (shared wobble)
    new_noise = np.random.normal(0, EDGE_JITTER * 0.2, NUM_PTS)
    jitter    = JITTER_SMOOTH * jitter + (1 - JITTER_SMOOTH) * new_noise
    jitter    = np.clip(jitter, -EDGE_JITTER, EDGE_JITTER)

    # 4) smooth by interpolation
    ang_ext     = np.concatenate([angles_base, angles_base + 2*np.pi])
    jit_ext     = np.concatenate([jitter, jitter])
    jitter_fine = np.interp(angles_fine, ang_ext, jit_ext)

    radii_major = a0 + jitter_fine
    radii_minor = b0 + jitter_fine

    # 5) contour bubble 1
    xs1  = cx1 + radii_major * np.cos(angles_fine)
    ys1  = cy1 + radii_minor * np.sin(angles_fine)
    pts1 = np.stack([xs1, ys1], axis=-1).astype(np.int32)[None, ...]

    contour1      = pts1[0].reshape(-1, 1, 2).astype(np.float32)
    current_area1 = cv2.contourArea(contour1)
    if current_area1 > 1e-3:
        scale1 = np.sqrt(TARGET_AREA / current_area1)
        xs1 = cx1 + (xs1 - cx1) * scale1
        ys1 = cy1 + (ys1 - cy1) * scale1
        pts1 = np.stack([xs1, ys1], axis=-1).astype(np.int32)[None, ...]

    # 6) contour bubble 2
    pts2 = None
    if has_bubble2 and (-100 <= cy2 <= h + 200):
        xs2  = cx2 + radii_major * np.cos(angles_fine)
        ys2  = cy2 + radii_minor * np.sin(angles_fine)
        pts2 = np.stack([xs2, ys2], axis=-1).astype(np.int32)[None, ...]

        contour2      = pts2[0].reshape(-1, 1, 2).astype(np.float32)
        current_area2 = cv2.contourArea(contour2)
        if current_area2 > 1e-3:
            scale2 = np.sqrt(TARGET_AREA / current_area2)
            xs2 = cx2 + (xs2 - cx2) * scale2
            ys2 = cy2 + (ys2 - cy2) * scale2
            pts2 = np.stack([xs2, ys2], axis=-1).astype(np.int32)[None, ...]

    # 7) build per-bubble masks + combined mask
    mask1 = np.zeros((h, w), np.uint8)
    cv2.fillPoly(mask1, pts1, 255)

    mask2 = np.zeros((h, w), np.uint8)
    if pts2 is not None:
        cv2.fillPoly(mask2, pts2, 255)

    # restrict bubbles to apparatus windows only
    left_window_mask = np.zeros((h, w), np.uint8)
    right_window_mask = np.zeros((h, w), np.uint8)
    cv2.rectangle(left_window_mask, (left_win_x0, win_y0), (left_win_x1, win_y1), 255, -1)
    cv2.rectangle(right_window_mask, (right_win_x0, win_y0), (right_win_x1, win_y1), 255, -1)

    if bidx % 2 == 0:
        mask1 = cv2.bitwise_and(mask1, left_window_mask)
        mask2 = cv2.bitwise_and(mask2, right_window_mask)
    else:
        mask1 = cv2.bitwise_and(mask1, right_window_mask)
        mask2 = cv2.bitwise_and(mask2, left_window_mask)

    mask = cv2.bitwise_or(mask1, mask2)

    # write GT mask video frame (combined)
    mask_out.write(mask)

    # composite bubbles into frame
    alpha3 = (mask.astype(float) / 255.0)[..., None] * BUBBLE_ALPHA
    layer  = np.full_like(frame, BUBBLE_COLOR)
    frame  = (frame * (1 - alpha3) + layer * alpha3).astype(np.uint8)

    # 8) glare for both bubbles
    ang_g = 2*np.pi * (i / NUM_FRAMES)
    glare = np.zeros((h, w), np.uint8)
    
    gx1 = int(cx1 + a0 * GLARE_MOVE_R * np.cos(ang_g))
    gy1 = int(cy1 + b0 * GLARE_MOVE_R * np.sin(ang_g))
    cv2.circle(glare, (gx1, gy1), GLARE_RADIUS, 255, -1)
    
    if has_bubble2 and cx2 is not None and cy2 is not None:
        gx2 = int(cx2 + a0 * GLARE_MOVE_R * np.cos(ang_g + np.pi/6))
        gy2 = int(cy2 + b0 * GLARE_MOVE_R * np.sin(ang_g + np.pi/6))
        cv2.circle(glare, (gx2, gy2), GLARE_RADIUS, 255, -1)
    
    glare = cv2.GaussianBlur(glare, (GLARE_BLUR, GLARE_BLUR), 0)
    ga3   = (glare.astype(float) / 255.0)[..., None] * GLARE_ALPHA
    frame = (frame * (1 - ga3) + 255 * ga3).astype(np.uint8)

    # 9) annotate
    top_txt = f"Frames per burst: {BURST_FRAMES}"
    cv2.putText(frame, top_txt, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                (0, 0, 0), 2, cv2.LINE_AA)

    info_txt = f"ID 0/1   D={CONST_DIAM:.1f}px   side={'L/R' if bidx % 2 == 0 else 'R/L'}   dx={burst_dx}"
    cv2.putText(frame, info_txt, (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                (0, 255, 0), 2, cv2.LINE_AA)

    # 10) black bar for timestamp
    bar_h = 30
    cv2.rectangle(frame, (0, h - bar_h), (w, h), (0, 0, 0), -1)

    # 11) Phantom-style timestamp overlay from time_ms
    ts_txt = f"FI+: +{time_ms:0.3f} ms"
    chars  = [c for c in ts_txt if c != " "]

    widths   = [char_templates[char_map.get(c, c)].shape[1] for c in chars]
    spacings = [kern(chars[j], chars[j+1]) for j in range(len(chars) - 1)]
    Wts      = int(sum(widths) + sum(spacings))
    Hts      = int(max(char_templates[char_map.get(c, c)].shape[0] for c in chars))

    overlay = np.zeros((Hts, Wts, 4), dtype=np.uint8)
    x_off   = 0
    for j, c in enumerate(chars):
        key = char_map.get(c, c)
        tpl = char_templates[key]
        hh, ww = tpl.shape[:2]
        overlay[0:hh, x_off:x_off + ww] = tpl
        x_off += ww
        if j < len(chars) - 1:
            x_off += kern(c, chars[j+1])

    X_ts = 5
    Y_ts = h - bar_h + (bar_h - Hts) // 2

    alpha = overlay[..., 3:4].astype(float) / 255.0
    roi   = frame[Y_ts:Y_ts + Hts, X_ts:X_ts + Wts].astype(float)
    comp  = roi * (1 - alpha) + overlay[..., :3].astype(float) * alpha
    frame[Y_ts:Y_ts + Hts, X_ts:X_ts + Wts] = comp.astype(np.uint8)

    # ── GT stats rows (bubble 0 = mask1, bubble 1 = mask2) ─────────────────
    s1 = gt_stats_from_mask(mask1)
    s2 = gt_stats_from_mask(mask2)

    # GT velocities must also flip sign when viewed from opposite side
    vx0_raw = float(2 * np.pi * 0.18 * x_amp * np.cos(2 * np.pi * 0.18 * t1_s))
    vy0 = 0.0

    if bidx % 2 == 0:
        vx0 = vx0_raw
    else:
        vx0 = -vx0_raw

    if has_bubble2:
        vx1_raw = float(2 * np.pi * 0.18 * x_amp * np.cos(2 * np.pi * 0.18 * t2_s))
        vy1 = 0.0
        if bidx % 2 == 0:
            vx1 = vx1_raw
        else:
            vx1 = -vx1_raw
    else:
        vx1, vy1 = float("nan"), float("nan")

    sp0 = float(np.sqrt(vx0*vx0 + vy0*vy0))
    sp1 = float(np.sqrt(vx1*vx1 + vy1*vy1)) if has_bubble2 else float("nan")

    gt_rows.append([
        i, time_ms / 1000.0,
        BURST_FRAMES, bidx, ib,

        # bubble 0 (always present)
        0, s1["cx"], s1["cy"], s1["area"], s1["diam"],
        vx0, vy0, sp0, s1["xmin"], s1["ymin"], s1["xmax"], s1["ymax"],

        # bubble 1 (may be absent/offscreen early)
        1, s2["cx"], s2["cy"], s2["area"], s2["diam"],
        vx1, vy1, sp1, s2["xmin"], s2["ymin"], s2["xmax"], s2["ymax"],
    ])

    # write frame + progress
    vid_out.write(frame)
    if (i + 1) % 50 == 0:
        print(f"frame {i+1}/{NUM_FRAMES}")

vid_out.release()
mask_out.release()

# write GT CSV
gt_csv_path = os.path.join(OUT_DIR, GT_CSV_NAME)
with open(gt_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "frame","time_s",
        "burst_frames","burst_index","inburst_index",

        "gt_id0","gt0_cx_px","gt0_cy_px","gt0_area_px2","gt0_diam_equiv_px",
        "gt0_vx_px_s","gt0_vy_px_s","gt0_speed_px_s",
        "gt0_bbox_xmin","gt0_bbox_ymin","gt0_bbox_xmax","gt0_bbox_ymax",

        "gt_id1","gt1_cx_px","gt1_cy_px","gt1_area_px2","gt1_diam_equiv_px",
        "gt1_vx_px_s","gt1_vy_px_s","gt1_speed_px_s",
        "gt1_bbox_xmin","gt1_bbox_ymin","gt1_bbox_xmax","gt1_bbox_ymax",
    ])
    writer.writerows(gt_rows)

print("Done.")
print("Video:", os.path.join(OUT_DIR, VID_NAME))
print("GT mask video:", os.path.join(OUT_DIR, MASK_NAME))
print("GT CSV:", gt_csv_path)


frame 50/500
frame 100/500
frame 150/500
frame 200/500
frame 250/500
frame 300/500
frame 350/500
frame 400/500
frame 450/500
frame 500/500
Done.
Video: synthetic_bubble\ellipse_bubble_wobbly_spin_bursts_two_bubbles.mp4
GT mask video: synthetic_bubble\ellipse_bubble_wobbly_spin_bursts_two_bubbles_gt_mask.mp4
GT CSV: synthetic_bubble\ellipse_bubble_wobbly_spin_bursts_two_bubbles_gt.csv


#### Muliple Bubbles

In [12]:
import os
import glob
import cv2
import numpy as np
import pandas as pd

# ── Load Phantom-style timestamp glyphs ──────────────────────────────────
def load_char_templates(dirname="templates"):
    templates = {}
    for p in glob.glob(os.path.join(dirname, "*.png")):
        name = os.path.splitext(os.path.basename(p))[0]
        img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise FileNotFoundError(f"Missing template: {p}")
        if img.shape[2] == 3:
            img = np.dstack([img, np.ones(img.shape[:2], np.uint8) * 255])
        templates[name] = img
    return templates

char_templates = load_char_templates("templates")
char_map = {".": "dot", ":": "colon", " ": "space"}

# ── CONFIG ───────────────────────────────────────────────────────────────
OUT_DIR  = "multi_bubble"
BASE     = "multi_bubble"
FPS      = 30
N_FRAMES = 500
W, H     = 1280, 832

VID_MAIN = f"{BASE}.mp4"
VID_MASK = f"{BASE}_masks.mp4"
CSV_PAR  = f"{BASE}_params.csv"
CSV_GT   = f"{BASE}_gt.csv"

# styling
BG    = 230
NOISE = 3
ALPHA = 0.8
COLOR = (30, 30, 30)

# glare
GLARE_R       = 7
GLARE_BLUR    = 15
GLARE_A       = 0.9
GLARE_MOVE_R  = 0.10  # fraction of radius

BAR_H = 30

# repeatable RNG
SEED = 12345
rng = np.random.default_rng(SEED)

ANGLES = np.linspace(0, 2*np.pi, 128, endpoint=False)

# ── bubble population (DETERMINISTIC, 12 TOTAL, EXACTLY 1 OVERLAP PAIR) ──
N_BUB = 12
types = ["circle", "ellipse", "wobble"]  # circle, ellipse, wobbly ellipse

objects = []
params  = []


SEED = 12345
rng = np.random.default_rng(SEED)

# Overlap target: near middle of video
k_catch = 250  # frames (middle of 500)
vy_s = -1.7
vy_f = -2.7
delta_y = (vy_s - vy_f) * k_catch  # cy0_f - cy0_s

lane_x_overlap = 650.0

# 10 non-overlap bubbles: widely separated x so they can't touch
# (keep at least ~150 px separation; these are ~80-120 px bubbles max)
x_nonoverlap = [200.0, 340.0, 480.0, 620.0, 760.0, 900.0, 1040.0, 1180.0, 260.0, 980.0]

# deterministic sizes (12 entries)
D_list = [60, 60, 40, 48, 70, 33, 44, 58, 36, 66, 52, 46]

# deterministic ellipse aspect ratios (for ellipse + wobble)
aspect_list = [1.0, 0.85, 0.75, 0.9, 0.65, 0.8, 0.7, 0.88, 0.72, 0.78, 0.68, 0.82]

# Non-overlap bubbles all move at same speed so none catch each other
vy_non = [-1.4, -1.6, -1.8, -2.0, -2.2, -2.4, -1.5, -1.7, -2.1, -2.3]

# Stagger their starting y so they appear at different times / heights, but never catch
cy_non = [H + 60.0, H + 110.0, H + 160.0, H + 210.0, H + 260.0,
          H + 310.0, H + 360.0, H + 410.0, H + 460.0, H + 510.0]

for i in range(N_BUB):
    D = float(D_list[i])
    aspect = float(aspect_list[i])

    # defaults for everyone (deterministic)
    vx_pf = 0.0
    phase = 0.20
    swayA = 0.0  # <-- IMPORTANT: no sideways sway for non-overlap bubbles

    # ---- Overlap pair: ids 0 and 1 ONLY ----
    if i == 0:
        cx0 = lane_x_overlap
        cy0 = H + 90.0
        vy_pf = vy_s
        swayA = 6.0     # allow a bit of sway, but same phase so they stay aligned
        phase = 0.20
    elif i == 1:
        cx0 = lane_x_overlap + 5.0
        cy0 = (H + 90.0) + float(delta_y)
        vy_pf = vy_f
        swayA = 6.0
        phase = 0.20
    else:
        # ---- All other bubbles: spaced out in x, same speed, no sway ----
        j = i - 2
        cx0 = float(x_nonoverlap[j])
        cy0 = float(cy_non[j])
        vy_pf = float(vy_non[j])
        # give them different phases anyway (doesn't matter since swayA=0)
        phase = float(0.5 + 0.3 * j)

    obj = {
        "id": i,
        "type": types[i % 3],
        "D": D,
        "a": D / 2.0,
        "b": (D / 2.0) * aspect,
        "cx0": cx0,
        "cy0": cy0,
        "vx_pf": vx_pf,
        "vy_pf": vy_pf,
        "phase": phase,
        "jitter": np.zeros(8, dtype=float),
        "swayA": float(swayA),
    }

    objects.append(obj)

    params.append({
        "id": i,
        "type": obj["type"],
        "diam_px": obj["D"],
        "vel_px_s": float(np.hypot(obj["vx_pf"], obj["vy_pf"]) * FPS),
        "vx_px_s": float(obj["vx_pf"] * FPS),
        "vy_px_s": float(obj["vy_pf"] * FPS),
    })

# ── timestamp drawing ────────────────────────────────────────────────────
def draw_timestamp_in_bar(frame, t_ms):
    txt = f"FI+: +{t_ms:0.3f} ms"
    chars = [c for c in txt if c != " "]

    widths = []
    heights = []
    for c in chars:
        key = char_map.get(c, c)
        if key not in char_templates:
            raise KeyError(f"Missing template for '{c}' (key '{key}')")
        h0, w0 = char_templates[key].shape[:2]
        widths.append(w0)
        heights.append(h0)

    gap = 2
    Wt = sum(widths) + gap * (len(chars) - 1)
    Ht = max(heights)

    overlay = np.zeros((Ht, Wt, 4), np.uint8)
    x = 0
    for j, c in enumerate(chars):
        key = char_map.get(c, c)
        tpl = char_templates[key]
        h0, w0 = tpl.shape[:2]
        overlay[:h0, x:x+w0] = tpl
        x += w0 + (gap if j < len(chars)-1 else 0)

    x0 = 5
    y0 = (H - BAR_H) + (BAR_H - Ht)//2

    alpha = overlay[..., 3:4].astype(np.float32) / 255.0
    roi = frame[y0:y0+Ht, x0:x0+Wt].astype(np.float32)
    comp = roi * (1 - alpha) + overlay[..., :3].astype(np.float32) * alpha
    frame[y0:y0+Ht, x0:x0+Wt] = comp.astype(np.uint8)

# ── writers ──────────────────────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

vid   = cv2.VideoWriter(os.path.join(OUT_DIR, VID_MAIN), fourcc, FPS, (W, H))
vid_m = cv2.VideoWriter(os.path.join(OUT_DIR, VID_MASK), fourcc, FPS, (W, H), False)

gt_rows = []

# ── main loop ────────────────────────────────────────────────────────────
for k in range(N_FRAMES):
    frame = np.full((H, W, 3), BG, np.uint8)

    # repeatable noise: rng is seeded and used deterministically
    noise = rng.normal(0, NOISE, frame.shape).astype(np.float32)
    frame = np.clip(frame.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    mask_all = np.zeros((H, W), np.uint8)
    t_ms = k * (1000.0 / FPS)

    for o in objects:
        # motion model (same structure, but sway amplitude controlled)
        t_s = k / FPS
        cx = o["cx0"] + o["vx_pf"] * k + o.get("swayA", 0.0) * np.sin(0.2 * t_s + o["phase"])
        cy = o["cy0"] + o["vy_pf"] * k

        if cy < -150 or cy > H + 300:
            continue

        # contour points
        if o["type"] == "circle":
            xs = cx + o["a"] * np.cos(ANGLES)
            ys = cy + o["a"] * np.sin(ANGLES)
            approx_r = o["a"]
        elif o["type"] == "ellipse":
            xs = cx + o["a"] * np.cos(ANGLES)
            ys = cy + o["b"] * np.sin(ANGLES)
            approx_r = 0.5 * (o["a"] + o["b"])
        else:  # wobble
            n = rng.normal(0, 4, 8)
            o["jitter"] = 0.9 * o["jitter"] + 0.1 * n
            base = np.linspace(0, 2*np.pi, 8, endpoint=False)
            jit = np.interp(
                ANGLES,
                np.r_[base, base + 2*np.pi],
                np.r_[o["jitter"], o["jitter"]]
            )
            xs = cx + (o["a"] + jit) * np.cos(ANGLES)
            ys = cy + (o["b"] + jit) * np.sin(ANGLES)
            approx_r = 0.5 * (o["a"] + o["b"])

        cnt = np.stack([xs, ys], axis=-1).astype(np.int32)[None, :, :]
        m = np.zeros((H, W), np.uint8)
        cv2.fillPoly(m, cnt, 255)

        # composite bubble fill
        mask_all |= m
        a3 = (m.astype(np.float32) / 255.0)[..., None] * ALPHA
        frame = (frame.astype(np.float32) * (1 - a3) + np.array(COLOR, np.float32) * a3).astype(np.uint8)

        # glare on top
        ang = 2 * np.pi * (k / N_FRAMES)
        gx = int(cx + approx_r * GLARE_MOVE_R * np.cos(ang))
        gy = int(cy + approx_r * GLARE_MOVE_R * np.sin(ang))

        glare = np.zeros((H, W), np.uint8)
        cv2.circle(glare, (gx, gy), GLARE_R, 255, -1)
        glare = cv2.GaussianBlur(glare, (GLARE_BLUR, GLARE_BLUR), 0)

        ga = (glare.astype(np.float32) / 255.0)[..., None] * GLARE_A
        frame = (frame.astype(np.float32) * (1 - ga) + 255.0 * ga).astype(np.uint8)

        # --- THESIS-STYLE GT ROW (unchanged) ---
        M = cv2.moments(m, binaryImage=True)
        if M["m00"] > 1e-6:
            gt_cx = float(M["m10"] / M["m00"])
            gt_cy = float(M["m01"] / M["m00"])
            area_px2 = int(cv2.countNonZero(m))
            gt_diam = float(np.sqrt(4.0 * area_px2 / np.pi))

            ys_nz, xs_nz = np.where(m > 0)
            xmin, xmax = int(xs_nz.min()), int(xs_nz.max())
            ymin, ymax = int(ys_nz.min()), int(ys_nz.max())
        else:
            gt_cx = gt_cy = float("nan")
            area_px2 = 0
            gt_diam = float("nan")
            xmin = ymin = xmax = ymax = -1

        gt_vx_px_s = float(o["vx_pf"] * FPS)
        gt_vy_px_s = float(o["vy_pf"] * FPS)
        gt_speed_px_s = float(np.hypot(gt_vx_px_s, gt_vy_px_s))

        gt_rows.append({
            "frame": k,
            "time_s": k / FPS,
            "gt_id": o["id"],
            "gt_cx_px": gt_cx,
            "gt_cy_px": gt_cy,
            "gt_area_px2": area_px2,
            "gt_diam_equiv_px": gt_diam,
            "gt_vx_px_s": gt_vx_px_s,
            "gt_vy_px_s": gt_vy_px_s,
            "gt_speed_px_s": gt_speed_px_s,
            "gt_bbox_xmin": xmin,
            "gt_bbox_ymin": ymin,
            "gt_bbox_xmax": xmax,
            "gt_bbox_ymax": ymax,
        })

    # bar + timestamp
    cv2.rectangle(frame, (0, H - BAR_H), (W, H), (0, 0, 0), -1)
    draw_timestamp_in_bar(frame, t_ms)

    vid.write(frame)
    vid_m.write(mask_all)

    if (k + 1) % 50 == 0:
        print(f"frame {k+1}/{N_FRAMES}")

vid.release()
vid_m.release()

pd.DataFrame(params).to_csv(os.path.join(OUT_DIR, CSV_PAR), index=False)
pd.DataFrame(gt_rows).to_csv(os.path.join(OUT_DIR, CSV_GT), index=False)

print("DONE →", os.path.join(OUT_DIR, VID_MAIN))
print("MASK →", os.path.join(OUT_DIR, VID_MASK))
print("PARAMS →", os.path.join(OUT_DIR, CSV_PAR))
print("GT CSV →", os.path.join(OUT_DIR, CSV_GT))

frame 50/500
frame 100/500
frame 150/500
frame 200/500
frame 250/500
frame 300/500
frame 350/500
frame 400/500
frame 450/500
frame 500/500
DONE → multi_bubble\multi_bubble.mp4
MASK → multi_bubble\multi_bubble_masks.mp4
PARAMS → multi_bubble\multi_bubble_params.csv
GT CSV → multi_bubble\multi_bubble_gt.csv


#### Multi Bubble Merged

In [13]:
import os
import glob
import cv2
import numpy as np
import pandas as pd

# ── Load Phantom-style timestamp glyphs ──────────────────────────────────
def load_char_templates(dirname="templates"):
    templates = {}
    for p in glob.glob(os.path.join(dirname, "*.png")):
        name = os.path.splitext(os.path.basename(p))[0]
        img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise FileNotFoundError(f"Missing template: {p}")
        if img.shape[2] == 3:
            img = np.dstack([img, np.ones(img.shape[:2], np.uint8) * 255])
        templates[name] = img
    return templates

char_templates = load_char_templates("templates")
char_map = {".": "dot", ":": "colon", " ": "space"}

# ── CONFIG ───────────────────────────────────────────────────────────────
OUT_DIR  = "multi_bubble"
BASE     = "merged_multi_bubble"
FPS      = 30
N_FRAMES = 500
W, H     = 1280, 832

VID_MAIN = f"{BASE}.mp4"
VID_MASK = f"{BASE}_masks.mp4"
CSV_PAR  = f"{BASE}_params.csv"
CSV_GT   = f"{BASE}_gt.csv"

# styling
BG    = 230
NOISE = 3
ALPHA = 0.8
COLOR = (30, 30, 30)

# glare
GLARE_R       = 7
GLARE_BLUR    = 15
GLARE_A       = 0.9
GLARE_MOVE_R  = 0.10  # fraction of radius

BAR_H = 30

# repeatable RNG
SEED = 12345
rng = np.random.default_rng(SEED)

ANGLES = np.linspace(0, 2*np.pi, 128, endpoint=False)

# --- GT merge settings for overlap pair only ---
MERGE_COV_TH = 0.70   # intersection / min(area0, area1)
MERGE_IOU_TH = 0.35   # intersection / union (backup)

def bbox_from_mask(m):
    ys, xs = np.where(m > 0)
    if xs.size == 0:
        return -1, -1, -1, -1
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def centroid_from_mask(m):
    M = cv2.moments(m, binaryImage=True)
    if M["m00"] <= 1e-9:
        return float("nan"), float("nan")
    return float(M["m10"] / M["m00"]), float(M["m01"] / M["m00"])

# ── bubble population (DETERMINISTIC, 12 TOTAL, EXACTLY 1 OVERLAP PAIR) ──
N_BUB = 12
types = ["circle", "ellipse", "wobble"]  # circle, ellipse, wobbly ellipse

objects = []
params  = []


SEED = 12345
rng = np.random.default_rng(SEED)

# Overlap target: near middle of video
k_catch = 250  # frames (middle of 500)
vy_s = -1.7
vy_f = -2.7
delta_y = (vy_s - vy_f) * k_catch  # cy0_f - cy0_s

lane_x_overlap = 650.0

# 10 non-overlap bubbles: widely separated x so they can't touch
# (keep at least ~150 px separation; these are ~80-120 px bubbles max)
x_nonoverlap = [200.0, 340.0, 480.0, 620.0, 760.0, 900.0, 1040.0, 1180.0, 260.0, 980.0]

# deterministic sizes (12 entries)
D_list = [60, 60, 40, 48, 70, 33, 44, 58, 36, 66, 52, 46]

# deterministic ellipse aspect ratios (for ellipse + wobble)
aspect_list = [1.0, 0.85, 0.75, 0.9, 0.65, 0.8, 0.7, 0.88, 0.72, 0.78, 0.68, 0.82]

# Non-overlap bubbles all move at same speed so none catch each other
vy_non = [-1.4, -1.6, -1.8, -2.0, -2.2, -2.4, -1.5, -1.7, -2.1, -2.3]

# Stagger their starting y so they appear at different times / heights, but never catch
cy_non = [H + 60.0, H + 110.0, H + 160.0, H + 210.0, H + 260.0,
          H + 310.0, H + 360.0, H + 410.0, H + 460.0, H + 510.0]

for i in range(N_BUB):
    D = float(D_list[i])
    aspect = float(aspect_list[i])

    # defaults for everyone (deterministic)
    vx_pf = 0.0
    phase = 0.20
    swayA = 0.0  # <-- IMPORTANT: no sideways sway for non-overlap bubbles

    # ---- Overlap pair: ids 0 and 1 ONLY ----
    if i == 0:
        cx0 = lane_x_overlap
        cy0 = H + 90.0
        vy_pf = vy_s
        swayA = 6.0     # allow a bit of sway, but same phase so they stay aligned
        phase = 0.20
    elif i == 1:
        cx0 = lane_x_overlap + 5.0
        cy0 = (H + 90.0) + float(delta_y)
        vy_pf = vy_f
        swayA = 6.0
        phase = 0.20
    else:
        # ---- All other bubbles: spaced out in x, same speed, no sway ----
        j = i - 2
        cx0 = float(x_nonoverlap[j])
        cy0 = float(cy_non[j])
        vy_pf = float(vy_non[j])
        # give them different phases anyway (doesn't matter since swayA=0)
        phase = float(0.5 + 0.3 * j)

    obj = {
        "id": i,
        "type": types[i % 3],
        "D": D,
        "a": D / 2.0,
        "b": (D / 2.0) * aspect,
        "cx0": cx0,
        "cy0": cy0,
        "vx_pf": vx_pf,
        "vy_pf": vy_pf,
        "phase": phase,
        "jitter": np.zeros(8, dtype=float),
        "swayA": float(swayA),
    }

    objects.append(obj)

    params.append({
        "id": i,
        "type": obj["type"],
        "diam_px": obj["D"],
        "vel_px_s": float(np.hypot(obj["vx_pf"], obj["vy_pf"]) * FPS),
        "vx_px_s": float(obj["vx_pf"] * FPS),
        "vy_px_s": float(obj["vy_pf"] * FPS),
    })

# ── timestamp drawing ────────────────────────────────────────────────────
def draw_timestamp_in_bar(frame, t_ms):
    txt = f"FI+: +{t_ms:0.3f} ms"
    chars = [c for c in txt if c != " "]

    widths = []
    heights = []
    for c in chars:
        key = char_map.get(c, c)
        if key not in char_templates:
            raise KeyError(f"Missing template for '{c}' (key '{key}')")
        h0, w0 = char_templates[key].shape[:2]
        widths.append(w0)
        heights.append(h0)

    gap = 2
    Wt = sum(widths) + gap * (len(chars) - 1)
    Ht = max(heights)

    overlay = np.zeros((Ht, Wt, 4), np.uint8)
    x = 0
    for j, c in enumerate(chars):
        key = char_map.get(c, c)
        tpl = char_templates[key]
        h0, w0 = tpl.shape[:2]
        overlay[:h0, x:x+w0] = tpl
        x += w0 + (gap if j < len(chars)-1 else 0)

    x0 = 5
    y0 = (H - BAR_H) + (BAR_H - Ht)//2

    alpha = overlay[..., 3:4].astype(np.float32) / 255.0
    roi = frame[y0:y0+Ht, x0:x0+Wt].astype(np.float32)
    comp = roi * (1 - alpha) + overlay[..., :3].astype(np.float32) * alpha
    frame[y0:y0+Ht, x0:x0+Wt] = comp.astype(np.uint8)

# ── writers ──────────────────────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

vid   = cv2.VideoWriter(os.path.join(OUT_DIR, VID_MAIN), fourcc, FPS, (W, H))
vid_m = cv2.VideoWriter(os.path.join(OUT_DIR, VID_MASK), fourcc, FPS, (W, H), False)

gt_rows = []

# ── main loop ────────────────────────────────────────────────────────────
for k in range(N_FRAMES):
    frame = np.full((H, W, 3), BG, np.uint8)

    # repeatable noise: rng is seeded and used deterministically
    noise = rng.normal(0, NOISE, frame.shape).astype(np.float32)
    frame = np.clip(frame.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    mask_all = np.zeros((H, W), np.uint8)
    t_ms = k * (1000.0 / FPS)

    bubble_obs = {}  

    for o in objects:
        # motion model (same structure, but sway amplitude controlled)
        t_s = k / FPS
        cx = o["cx0"] + o["vx_pf"] * k + o.get("swayA", 0.0) * np.sin(0.2 * t_s + o["phase"])
        cy = o["cy0"] + o["vy_pf"] * k

        if cy < -150 or cy > H + 300:
            continue

        # contour points
        if o["type"] == "circle":
            xs = cx + o["a"] * np.cos(ANGLES)
            ys = cy + o["a"] * np.sin(ANGLES)
            approx_r = o["a"]
        elif o["type"] == "ellipse":
            xs = cx + o["a"] * np.cos(ANGLES)
            ys = cy + o["b"] * np.sin(ANGLES)
            approx_r = 0.5 * (o["a"] + o["b"])
        else:  # wobble
            n = rng.normal(0, 4, 8)
            o["jitter"] = 0.9 * o["jitter"] + 0.1 * n
            base = np.linspace(0, 2*np.pi, 8, endpoint=False)
            jit = np.interp(
                ANGLES,
                np.r_[base, base + 2*np.pi],
                np.r_[o["jitter"], o["jitter"]]
            )
            xs = cx + (o["a"] + jit) * np.cos(ANGLES)
            ys = cy + (o["b"] + jit) * np.sin(ANGLES)
            approx_r = 0.5 * (o["a"] + o["b"])

        cnt = np.stack([xs, ys], axis=-1).astype(np.int32)[None, :, :]
        m = np.zeros((H, W), np.uint8)
        cv2.fillPoly(m, cnt, 255)

        # composite bubble fill
        mask_all |= m
        a3 = (m.astype(np.float32) / 255.0)[..., None] * ALPHA
        frame = (frame.astype(np.float32) * (1 - a3) + np.array(COLOR, np.float32) * a3).astype(np.uint8)

        # glare on top
        ang = 2 * np.pi * (k / N_FRAMES)
        gx = int(cx + approx_r * GLARE_MOVE_R * np.cos(ang))
        gy = int(cy + approx_r * GLARE_MOVE_R * np.sin(ang))

        glare = np.zeros((H, W), np.uint8)
        cv2.circle(glare, (gx, gy), GLARE_R, 255, -1)
        glare = cv2.GaussianBlur(glare, (GLARE_BLUR, GLARE_BLUR), 0)

        ga = (glare.astype(np.float32) / 255.0)[..., None] * GLARE_A
        frame = (frame.astype(np.float32) * (1 - ga) + 255.0 * ga).astype(np.uint8)

        # store per-bubble mask/stats for merge-aware GT
        bid = int(o["id"])
        bubble_obs[bid] = {
            "mask": m,  # uint8 0/255
            "area": int(cv2.countNonZero(m)),
            "vx": float(o["vx_pf"] * FPS),
            "vy": float(o["vy_pf"] * FPS),
        }
    # ---------------- MERGE-AWARE GT (only ids 0 and 1) ----------------
    def append_gt_row(gt_id, mask, vx, vy, n_members=1, member_ids=None):
        area_px2 = int(cv2.countNonZero(mask))
        gt_cx, gt_cy = centroid_from_mask(mask)
        xmin, ymin, xmax, ymax = bbox_from_mask(mask)
        gt_diam = float(np.sqrt(4.0 * area_px2 / np.pi)) if area_px2 > 0 else float("nan")
        gt_speed = float(np.hypot(vx, vy)) if np.isfinite(vx) and np.isfinite(vy) else float("nan")

        row = {
            "frame": k,
            "time_s": k / FPS,
            "gt_id": int(gt_id),
            "gt_cx_px": float(gt_cx),
            "gt_cy_px": float(gt_cy),
            "gt_area_px2": int(area_px2),
            "gt_diam_equiv_px": float(gt_diam),
            "gt_vx_px_s": float(vx),
            "gt_vy_px_s": float(vy),
            "gt_speed_px_s": float(gt_speed),
            "gt_bbox_xmin": int(xmin),
            "gt_bbox_ymin": int(ymin),
            "gt_bbox_xmax": int(xmax),
            "gt_bbox_ymax": int(ymax),
            "gt_n_members": int(n_members),
            "gt_member_ids": "" if not member_ids else ",".join(map(str, member_ids)),
        }
        gt_rows.append(row)

    merge_pair = (0 in bubble_obs) and (1 in bubble_obs)
    merged_now = False
    m01 = None

    if merge_pair:
        m0 = bubble_obs[0]["mask"]
        m1 = bubble_obs[1]["mask"]
        a0 = bubble_obs[0]["area"]
        a1 = bubble_obs[1]["area"]

        if a0 > 0 and a1 > 0:
            inter = int(cv2.countNonZero(cv2.bitwise_and(m0, m1)))
            if inter > 0:
                union = a0 + a1 - inter
                iou = inter / union if union > 0 else 0.0
                cov = inter / min(a0, a1)

                if (cov >= MERGE_COV_TH) or (iou >= MERGE_IOU_TH):
                    merged_now = True
                    m01 = cv2.bitwise_or(m0, m1)

    if merged_now:
        # area-weighted mean velocity for merged blob
        a0 = bubble_obs[0]["area"]
        a1 = bubble_obs[1]["area"]
        vx = (bubble_obs[0]["vx"] * a0 + bubble_obs[1]["vx"] * a1) / (a0 + a1)
        vy = (bubble_obs[0]["vy"] * a0 + bubble_obs[1]["vy"] * a1) / (a0 + a1)

        append_gt_row(gt_id=10001, mask=m01, vx=vx, vy=vy, n_members=2, member_ids=[0, 1])

        for bid, info in bubble_obs.items():
            if bid in (0, 1):
                continue
            append_gt_row(gt_id=bid, mask=info["mask"], vx=info["vx"], vy=info["vy"], n_members=1, member_ids=[bid])
    else:
        for bid, info in bubble_obs.items():
            append_gt_row(gt_id=bid, mask=info["mask"], vx=info["vx"], vy=info["vy"], n_members=1, member_ids=[bid])
            

    # bar + timestamp
    cv2.rectangle(frame, (0, H - BAR_H), (W, H), (0, 0, 0), -1)
    draw_timestamp_in_bar(frame, t_ms)

    vid.write(frame)
    vid_m.write(mask_all)

    if (k + 1) % 50 == 0:
        print(f"frame {k+1}/{N_FRAMES}")

    
vid.release()
vid_m.release()

pd.DataFrame(params).to_csv(os.path.join(OUT_DIR, CSV_PAR), index=False)
pd.DataFrame(gt_rows).to_csv(os.path.join(OUT_DIR, CSV_GT), index=False)

print("DONE →", os.path.join(OUT_DIR, VID_MAIN))
print("MASK →", os.path.join(OUT_DIR, VID_MASK))
print("PARAMS →", os.path.join(OUT_DIR, CSV_PAR))
print("GT CSV →", os.path.join(OUT_DIR, CSV_GT))

frame 50/500
frame 100/500
frame 150/500
frame 200/500
frame 250/500
frame 300/500
frame 350/500
frame 400/500
frame 450/500
frame 500/500
DONE → multi_bubble\merged_multi_bubble.mp4
MASK → multi_bubble\merged_multi_bubble_masks.mp4
PARAMS → multi_bubble\merged_multi_bubble_params.csv
GT CSV → multi_bubble\merged_multi_bubble_gt.csv


#### Synthetic Training Data

In [10]:
import os
import json
import cv2
import numpy as np
import math
from tqdm import tqdm
import glob

# ── CONFIG ─────────────────────────────────────────────────────────────
OUT_BASE     = "synthetic_dataset"
TRAIN_N      = 1000
VAL_N        = 200
FRAME_SIZE   = (1280, 832)
FPS          = 30
MIN_BUBBLES  = 1
MAX_BUBBLES  = 20
MIN_DIAMETER = 25   # px
MAX_DIAMETER = 120  # px
BAR_H        = 30
BKG_BRIGHT   = 255
NOISE_SIGMA  = 3
BUBBLE_ALPHA = 0.7
GLARE_RADIUS = 7
GLARE_BLUR   = 15
GLARE_MOVE_R = 0.1
GLARE_ALPHA  = 0.9
TYPES        = ["circle", "ellipse", "wobble"]
ENTRY_PROB   = 0.3
ENTRY_FRAC   = [0.1, 0.25, 0.5, 0.75]
PATCH_SIZE   = 128

# wobble params
NUM_PTS      = 8
FINE_FACTOR  = 16
EDGE_JITTER  = 5
angles_base  = np.linspace(0, 2*np.pi, NUM_PTS, endpoint=False)
angles_fine  = np.linspace(0, 2*np.pi, NUM_PTS*FINE_FACTOR, endpoint=False)

# RNG
rng = np.random.default_rng()

# ── Timestamp loader & drawer ─────────────────────────────
char_templates = {}
def load_char_templates(dirname="templates"):
    for p in glob.glob(os.path.join(dirname, "*.png")):
        key = os.path.splitext(os.path.basename(p))[0]
        img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise FileNotFoundError(f"Missing template: {p}")
        if img.shape[2] == 3:
            bgr = img
            a = 255 * np.ones(img.shape[:2], np.uint8)
            img = np.dstack([bgr, a])
        char_templates[key] = img
load_char_templates()
def draw_timestamp(frame, frame_idx):
    h, w = frame.shape[:2]
    txt = f"FI+: +{frame_idx*(1000.0/FPS):0.3f} ms"
    chars = list(txt)
    def kern(p,n):
        if p==":" and n=="+": return 4
        if n=="m": return 5
        if p=="1": return 2
        return 1
    widths = [char_templates.get(c,char_templates['space']).shape[1] for c in chars]
    spaces = [kern(chars[i],chars[i+1]) for i in range(len(chars)-1)] + [0]
    Wt = sum(widths) + sum(spaces)
    Ht = max(char_templates.get(c,char_templates['space']).shape[0] for c in chars)
    ov = np.zeros((Ht, Wt, 4), np.uint8)
    x0 = 0
    for i,c in enumerate(chars):
        tpl = char_templates.get(c,char_templates['space'])
        h0,w0 = tpl.shape[:2]
        ov[0:h0, x0:x0+w0] = tpl
        x0 += w0 + spaces[i]
    X_ts = 10
    Y_ts = h - BAR_H + (BAR_H - Ht)//2
    alpha = ov[...,3:4].astype(float)/255.0
    roi = frame[Y_ts:Y_ts+Ht, X_ts:X_ts+Wt].astype(float)
    frame[Y_ts:Y_ts+Ht, X_ts:X_ts+Wt] = ((1-alpha)*roi + alpha*ov[...,:3].astype(float)).astype(np.uint8)

# ── Generate images + COCO annotations ─────────────────────────────────
def generate(n, split):
    coco = {"images":[], "annotations":[], "categories":[{"id":1,"name":"bubble"}]}
    img_id = 0
    ann_id = 1
    h_img, w_img = FRAME_SIZE[::-1]
    bar_top = h_img - BAR_H
    layer = np.full((h_img, w_img, 3), (30,30,30), np.uint8)

    for j in tqdm(range(n), desc=split):
        # base frame
        frame = (np.ones((h_img, w_img, 3), dtype=np.float32)*BKG_BRIGHT +
                 np.random.normal(0, NOISE_SIGMA, (h_img, w_img, 3))).clip(0,255).astype(np.uint8)
        bubbles = []  # store (raw_mask, cx, cy, a, b)
        # draw bubbles
        for _ in range(int(rng.integers(MIN_BUBBLES, MAX_BUBBLES+1))):
            shape = rng.choice(TYPES)
            diam  = float(rng.uniform(MIN_DIAMETER, MAX_DIAMETER))
            a = diam/2; b={'circle':a,'ellipse':a*0.6,'wobble':a*0.8}[shape]
            cx = int(rng.uniform(a, w_img-a))
            cy = (int(bar_top - rng.choice(ENTRY_FRAC)*(2*b) + b)
                  if rng.random()<ENTRY_PROB else int(rng.uniform(a, bar_top-a)))
            # raw mask
            mask = np.zeros((h_img, w_img), np.uint8)
            if shape=='circle': cv2.circle(mask,(cx,cy),int(a),255,-1)
            elif shape=='ellipse': cv2.ellipse(mask,(cx,cy),(int(a),int(b)),0,0,360,255,-1)
            else:
                jit = rng.normal(0, EDGE_JITTER, NUM_PTS)
                ext_ang, ext_j = np.concatenate([angles_base,angles_base+2*np.pi]), np.concatenate([jit,jit])
                jit_f = np.interp(angles_fine, ext_ang, ext_j)
                xs = cx + (a+jit_f)*np.cos(angles_fine)
                ys = cy + (b+jit_f)*np.sin(angles_fine)
                pts=np.stack([xs,ys],axis=-1).astype(np.int32)[None,...]
                cv2.fillPoly(mask,pts,255)
            bubbles.append((mask,cx,cy,a,b))
            # composite & glare
            alpha = (mask[...,None]/255.0)*BUBBLE_ALPHA
            frame = (frame*(1-alpha) + layer*alpha).astype(np.uint8)
            ang = 2*np.pi*(j/float(n))
            gx,gy = int(cx+a*GLARE_MOVE_R*math.cos(ang)), int(cy+b*GLARE_MOVE_R*math.sin(ang))
            g = np.zeros((h_img,w_img),np.uint8); cv2.circle(g,(gx,gy),GLARE_RADIUS,255,-1)
            gblur = cv2.GaussianBlur(g,(GLARE_BLUR,GLARE_BLUR),0)
            alpha_g = (gblur[...,None]/255.0)*GLARE_ALPHA
            frame = (frame*(1-alpha_g) + 255*alpha_g).astype(np.uint8)
        # overlay bar + timestamp
        cv2.rectangle(frame,(0,bar_top),(w_img,h_img),(0,0,0),-1)
        draw_timestamp(frame,j)
        # save image entry
        os.makedirs(f"{OUT_BASE}/{split}/images",exist_ok=True)
        fn=f"{img_id:05d}.png"; cv2.imwrite(f"{OUT_BASE}/{split}/images/{fn}",frame)
        coco["images"].append({"id":img_id,"width":w_img,"height":h_img,"file_name":fn})
        # annotate each bubble mask clipped above the bar
        for mask,cx,cy,a,b in bubbles:
            mask_vis = mask.copy(); mask_vis[bar_top:,:]=0
            ys,xs = np.where(mask_vis>0)
            if ys.size==0: continue
            # find contours on visible mask
            cnts,_ = cv2.findContours(mask_vis,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            if cnts:
                c = max(cnts,key=cv2.contourArea)
                area = float(cv2.contourArea(c))
                x,y,wb,hb = cv2.boundingRect(c)
                seg_pts = c.reshape(-1,2)
            else:
                # fallback: bounding box polygon
                y0,y1 = ys.min(), ys.max(); x0,x1 = xs.min(), xs.max()
                area = float((y1-y0+1)*(x1-x0+1))
                x,y,wb,hb,seg_pts = x0,y0,x1-x0,y1-y0, np.array([[x0,y0],[x1,y0],[x1,y1],[x0,y1]])
            # flatten segmentation
            seg_flat = seg_pts.flatten().tolist()
            # only include if valid (>=6 coords)
            if len(seg_flat)>=6:
                coco["annotations"].append({
                    "id":ann_id,
                    "image_id":img_id,
                    "category_id":1,
                    "segmentation":[seg_flat],
                    "area":area,
                    "bbox":[x,y,wb,hb],
                    "iscrowd":0
                })
                ann_id+=1
        img_id+=1
    with open(f"{OUT_BASE}/{split}/annotations.json","w") as f: json.dump(coco,f)
    print(f"Wrote COCO → {OUT_BASE}/{split}/annotations.json")
    return coco

# ── Extract 128×128 patches for CNN ────────────────────────────────────
def extract_patches(coco, split):
    pos_dir,neg_dir = f"cnn_data/{split}/pos",f"cnn_data/{split}/neg"
    os.makedirs(pos_dir,exist_ok=True); os.makedirs(neg_dir,exist_ok=True)
    ann_map={}
    for ann in coco["annotations"]: ann_map.setdefault(ann["image_id"],[]).append(ann)
    for imd in tqdm(coco["images"],desc=f"Patching {split}"):
        img=cv2.imread(f"{OUT_BASE}/{split}/images/{imd['file_name']}")
        h,w=img.shape[:2]; mask=np.zeros((h,w),np.uint8)
        for ann in ann_map[imd['id']]:
            poly=np.array(ann['segmentation'][0],float).reshape(-1,2).round().astype(int)
            cv2.fillPoly(mask,[poly],255)
        x,y,wb,hb=ann_map[imd['id']][0]['bbox'];cx,cy=int(x+wb/2),int(y+hb/2)
        half=PATCH_SIZE//2; y1,y2=cy-half,cy+half; x1,x2=cx-half,cx+half
        y1c,y2c=max(0,y1),min(h,y2); x1c,x2c=max(0,x1),min(w,x2)
        patch=img[y1c:y2c,x1c:x2c]
        patch=cv2.copyMakeBorder(patch,y1c-y1,y2-y2c,x1c-x1,x2-x2c,cv2.BORDER_CONSTANT,value=[255]*3)
        cv2.imwrite(f"{pos_dir}/{imd['id']:05d}.png",patch)
        while True:
            nx,ny=rng.integers(half,w-half),rng.integers(half,h-half)
            if mask[ny-half:ny+half,nx-half:nx+half].sum()==0:
                neg=img[ny-half:ny+half,nx-half:nx+half]; cv2.imwrite(f"{neg_dir}/{imd['id']:05d}.png",neg)
                break

if __name__ == "__main__":
    ct=generate(TRAIN_N,"train"); cv=generate(VAL_N,"val")
    extract_patches(ct,"train"); extract_patches(cv,"val")
    print("All done!")

train:   0%|▎                                                                         | 4/1000 [00:06<27:21,  1.65s/it]


KeyboardInterrupt: 